In [3]:
import jax
import jax.numpy as jnp
import flax.nnx as nnx
import netket as nk
import netket.experimental as nkx
import sys
sys.path.append('..')
from NES_VMC import NESTotalAnsatz, create_machine,\
    ha,SingleStateAnsatz,create_single_machine,\
        create_machine_matrix,Ham_psi,Ham_Psi,NES_loss_energy,nes_vmc_gradient,hi,E_fcis,\
        NESFermionHopRule,compute_qgt,sampler_info
import optax
from typing import Callable
from functools import partial
from jax.flatten_util import ravel_pytree
import time

# ========== 你原有全局参数（直接复用） ==========
# 单系统希尔伯特空间
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,
    s=1/2,
    n_fermions_per_spin=(1,1),
)
K = 3  # NES 扩展副本数
hi_ext = hi ** K  # 扩展希尔伯特空间
SINGLE_SIZE = hi.size  # 单个子系统维度 = 4
single_edges = ((0, 1), (2, 3))  # 费米子跃迁边
g = nk.graph.Graph(edges=single_edges)
single_rule = nk.sampler.rules.FermionHopRule(hi, graph=g)
tensor_rule = nk.sampler.rules.TensorRule(hi_ext, [single_rule] * K)

total_ansatz = NESTotalAnsatz(4,K,12,rngs=nnx.Rngs(11))
total_machine, total_graphdef,total_params = create_machine(total_ansatz)
total_matrix_machine, total_graphdef,total_params = create_machine_matrix(total_ansatz)

single_machine_list = []
for ansatz in total_ansatz.single_ansatz_list:
    m, g, p = create_single_machine(ansatz)
    single_machine_list.append(m)

H₂ FCI 基准能量
E0 = -1.01546825 Ha  |  激发能：0.0000 eV
E1 = -0.87542794 Ha  |  激发能：3.8107 eV
E2 = -0.42938376 Ha  |  激发能：15.9482 eV
E3 = -0.26922131 Ha  |  激发能：20.3064 eV


In [4]:
import jax
import jax.numpy as jnp
import netket as nk

SINGLE_SIZE = hi.size

@nk.utils.struct.dataclass
class NESFermionHopRule(nk.sampler.rules.MetropolisRule):
    edges: jnp.ndarray
    K: int = nk.utils.struct.static_field()
    single_size: int = nk.utils.struct.static_field()

    def _check_duplicate(self, sigma_ext):
        """NES约束：检测任意两个子组态重复
        兼容一维单样本(返回标量) / 二维批量(返回batch数组)
        """
        one_d_input = (sigma_ext.ndim == 1)
        if one_d_input:
            sigma_ext = sigma_ext[None, :]
        
        batch_dim = sigma_ext.shape[0]
        sub = sigma_ext.reshape((batch_dim, self.K, self.single_size))
        # 全部子组态两两比对
        pair_equal = jnp.all(sub[:, :, None, :] == sub[:, None, :], axis=-1)
        diag_mask = jnp.eye(self.K, dtype=jnp.bool_)[None, :, :]
        off_diag_dup = jnp.where(diag_mask, False, pair_equal)
        batch_dup = jnp.any(off_diag_dup, axis=(-2, -1))
        
        if one_d_input:
            return batch_dup.squeeze()
        return batch_dup

    # 修复：补齐完整7个形参：self, sampler, machine, parameters, state, rng, sigma
    def transition(self, sampler, machine, parameters, state, rng, sigma):
        """跃迁规则"""
        batch_size = sigma.shape[0]
        key1, key2 = jax.random.split(rng)

        e_idx = jax.random.randint(key1, (batch_size,), 0, self.edges.shape[0])
        sel_e = self.edges[e_idx]
        i, j = sel_e[:,0], sel_e[:,1]

        sigma_cand = sigma.at[jnp.arange(batch_size),i].set(sigma[jnp.arange(batch_size),j])
        sigma_cand = sigma_cand.at[jnp.arange(batch_size),j].set(sigma[jnp.arange(batch_size),i])

        invalid = self._check_duplicate(sigma_cand)
        new_sigma = jnp.where(invalid[:, None], sigma, sigma_cand)

        return new_sigma, None

    def random_state(self, sampler, machine, parameters, state, rng):
        """随机态生成（完全不变）"""
        sigma_shape = state.σ.shape
        hilbert = sampler.hilbert

        def gen_single(key):
            max_tries = 100
            def cond(c): 
                return (c[0] < max_tries) & c[2]
            
            def body(c):
                tries, k, _, _ = c
                k, k_new = jax.random.split(k)
                s = hilbert.random_state(k_new)
                is_dup = self._check_duplicate(s)  # 一维输入自动返回标量
                return (tries + 1, k, is_dup, s)
            
            # 初始值 c[2] = True（标量布尔值），匹配while_loop
            init_c = (0, key, True, hilbert.random_state(key))
            final_c = jax.lax.while_loop(cond, body, init_c)
            tries, _, is_dup, s = final_c
            return jax.lax.cond(is_dup, lambda: hilbert.random_state(key), lambda: s)
        
        keys = jax.random.split(rng, sigma_shape[0])
        return jax.vmap(gen_single)(keys)

In [5]:
import logging
# 日志配置
logger = logging.getLogger('NES_VMC_K4')
logger.setLevel(logging.INFO)
# 阻止日志向上传播
logger.propagate = False
# 清除所有旧handler，防止重复打印
logger.handlers.clear()

# 自定义日志格式：只打印内容，不带等级、logger名
simple_formatter = logging.Formatter("%(message)s", datefmt="%H:%M:%S")

# 1. 文件输出处理器
file_handler = logging.FileHandler("nes_vmc_0616_K4.log", mode="w", encoding="utf-8")
file_handler.setFormatter(simple_formatter)
file_handler.setLevel(logging.INFO)
logger.addHandler(file_handler)

console_handler = logging.StreamHandler()
console_handler.setFormatter(simple_formatter)
console_handler.setLevel(logging.INFO)
logger.addHandler(console_handler)

print('库导入完成')

库导入完成


In [6]:
N_CHAINS = 16
N_WARMUP = 100
N_SAMPLES_PER_CHAIN = 200
SWEEP_SIZE = 30
N_ITER =3
SINGLE_SIZE = hi.size  # 单个子系统维度 = 4
Natural_Grad = False


total_ansatz = NESTotalAnsatz(4,K,12,rngs=nnx.Rngs(11))
total_machine, total_graphdef,total_params = create_machine(total_ansatz)
total_matrix_machine, total_graphdef,total_params = create_machine_matrix(total_ansatz)

single_machine_list = []
for ansatz in total_ansatz.single_ansatz_list:
    m, g, p = create_single_machine(ansatz)
    single_machine_list.append(m)
    
    

optimizer = optax.sgd(learning_rate=0.01)
opt_state = optimizer.init(total_params)

ext_edges = []
for k in range(K):
    offset = k * SINGLE_SIZE
    for (i, j) in single_edges:
        ext_edges.append((i + offset, j + offset))
ext_edges = jnp.array(ext_edges)  # 转为jax数组（关键修复）

nes_rule = NESFermionHopRule(edges=ext_edges, K=K, single_size=SINGLE_SIZE)
nes_sampler = nk.sampler.MetropolisSampler(
    hilbert=hi_ext,
    rule=nes_rule,
    n_chains=16,
    sweep_size=20
)


# 采样器状态初始化（替代原 init_sampler_state）
sampler_rng = jax.random.PRNGKey(21)
sampler_state = nes_sampler.init_state(total_machine, total_params, sampler_rng)

# ==================== 训练循环（仅替换采样部分） ====================
logger.info("\n" + "="*60)
logger.info("开始多链 NES-VMC 训练 (NetKet 自定义采样器 + 朴素梯度下降)")
logger.info("="*60)
logger.info(f"基态能量={E_fcis[0]:.8f} Ha| 第一激发态能量={E_fcis[1]:.8f} Ha| 第二激发态能量={E_fcis[2]:.8f} Ha| 第三激发态能量={E_fcis[3]:.8f} Ha")

history = {
    'step': [],
    'energy_0st': [],
    'energy_1st': [],
    'energy_2st': [],
    'energy_3st': [],
    'energy_std': [],
    'loss': [],
    'params': [],
    'E_Lmatrix':[],
    'natural_grad':[],
    'grad_flat':[],
    'samples':[],
    'log_Psi':[],
    'log_M':[],
    'log_Psi_mean':[],
    'log_Psi_min':[],
    'log_Psi_max':[],
    'grad_norm_before':[],
    'grad_norm_after':[],
}

start_time = time.time()
for step in range(N_ITER):
    # 2. 正式采样
    samples_raw, sampler_state = nes_sampler.sample(
        machine=total_machine, parameters=total_params, 
        state=sampler_state, chain_length=N_SAMPLES_PER_CHAIN
    )
        # 3. 维度重塑，适配梯度函数输入
    samples = samples_raw.reshape(-1, hi_ext.size)
    x_batch = samples.reshape(-1, K, 4)
    # 3. 计算能量和自然梯度（逻辑和原代码一致）
    grad, loss_mean, E_L_mean = nes_vmc_gradient(ha=ha,
                                                 total_matrix_machine=total_matrix_machine,
                                                 total_machine=total_machine,
                                                 single_machine_list=single_machine_list,
                                                 total_params=total_params,
                                                 x_batch=samples.reshape(-1,K,4))
    #grad = jax.tree_util.tree_map(lambda x: x * 2, grad)
    grad_flat , grad_unravel_fn = ravel_pytree(grad)
    grad_norm_before = jnp.linalg.norm(grad_flat)
    
    if Natural_Grad == True:

        qgt_reg, unravel_fn = compute_qgt(total_machine, total_params, samples.reshape(-1,K,4), diag_shift=0.01)
        natural_grad_flat = jnp.linalg.solve(qgt_reg, grad_flat)
        natural_grad = grad_unravel_fn(natural_grad_flat)
        grad = natural_grad
        
    # 4. 更新参数
    updates, opt_state = optimizer.update(grad, opt_state, total_params)
    total_params = optax.apply_updates(total_params, updates)

    log_Psi_batch = total_machine(total_params, samples.reshape(-1,K,4))
    eig_vals, eig_vecs = jnp.linalg.eigh(E_L_mean)
    
    grad_flat , grad_unravel_fn = ravel_pytree(grad)
    grad_norm_after = jnp.linalg.norm(grad_flat)
    
    
    history['step'].append(step)
    history['E_Lmatrix'].append(E_L_mean)
    history['samples'].append(samples)
    history['loss'].append(loss_mean)
    history['log_Psi_mean'].append(log_Psi_batch.mean())
    history['log_Psi_min'].append(log_Psi_batch.min())
    history['log_Psi_max'].append(log_Psi_batch.max())
    history['grad_norm_before'].append(grad_norm_before)
    history['grad_norm_after'].append(grad_norm_after)
    history['energy_0st'].append(eig_vals[0])
    history['energy_1st'].append(eig_vals[1])
    history['energy_2st'].append(eig_vals[2])
    history['energy_3st'].append(eig_vals[3])
    history['params'].append(total_params)
    # 5. 记录历史
    if step % 1 == 0 or step == N_ITER - 1:
        # --------------------- 【NES-VMC 监控模板】直接用 ---------------------
        # 1. 监控 log_Psi
        logger.info(f"log_Psi: mean={log_Psi_batch.mean():.3f} | min={log_Psi_batch.min():.3f} | max={log_Psi_batch.max():.3f}")
        # 2. 监控梯度范数
        logger.info(f"grad norm before = {grad_norm_before:.4f}|grad norm after = {grad_norm_after:.4f}")
        logger.info(f"Step {step:3d} | Loss: {loss_mean}|0st能量={eig_vals[0]:.8f} Ha｜1st能量={eig_vals[1]:.8f} Ha｜2st能量={eig_vals[2]:.8f} Ha｜3st能量={eig_vals[3]:.8f} Ha")
        # print(f'grad={grad_flat[30:31]}')
        logger.info('#-----------------------------------------#')


end_time = time.time()
print(f"训练耗时：{end_time - start_time:.2f} 秒")
# 最终结果
print("\n" + "="*60)
print(f"训练完成!")
print("="*60)


开始多链 NES-VMC 训练 (NetKet 自定义采样器 + 朴素梯度下降)
基态能量=-1.01546825 Ha| 第一激发态能量=-0.87542794 Ha| 第二激发态能量=-0.42938376 Ha| 第三激发态能量=-0.26922131 Ha
log_Psi: mean=0.972-0.708j | min=0.555-2.235j | max=1.046+1.135j
grad norm before = 2.9905|grad norm after = 2.9905
Step   0 | Loss: -1.9073312710365369|0st能量=-1.71035712 Ha｜1st能量=-0.72800487 Ha｜2st能量=0.53103072 Ha｜3st能量=0.53103072 Ha
#-----------------------------------------#
log_Psi: mean=0.925-0.720j | min=0.593-2.305j | max=1.090+0.996j
grad norm before = 2.3913|grad norm after = 2.3913
Step   1 | Loss: -2.1056469886285476|0st能量=-1.63599224 Ha｜1st能量=-0.82058771 Ha｜2st能量=0.35093296 Ha｜3st能量=0.35093296 Ha
#-----------------------------------------#
log_Psi: mean=0.970-0.734j | min=0.375-2.281j | max=1.096+0.931j
grad norm before = 1.2502|grad norm after = 1.2502
Step   2 | Loss: -2.1955600623671927|0st能量=-1.54974951 Ha｜1st能量=-0.85493082 Ha｜2st能量=0.20912027 Ha｜3st能量=0.20912027 Ha
#-----------------------------------------#


训练耗时：4.08 秒

训练完成!


$$
\begin{align*}
\Psi(\mathbf{x})^{-1}\hat{\mathcal{H}}\Psi(\mathbf{x})
&= \mathrm{Tr}\left[ \Psi^{-1}(\mathbf{x})\hat{H}\Psi(\mathbf{x}) \right]
\end{align*}
$$

In [7]:
counter = sampler_info(history['samples'][0],K)
abnormal_params = history['params'][0]
items, counts = zip(*counter.items())

元组 (0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1) 出现了 230 次
元组 (0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0) 出现了 147 次
元组 (1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1) 出现了 146 次
元组 (1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1) 出现了 385 次
元组 (1, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1) 出现了 251 次
元组 (1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1) 出现了 220 次
元组 (0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0) 出现了 65 次
元组 (0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0) 出现了 103 次
元组 (0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0) 出现了 50 次
元组 (1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1) 出现了 147 次
元组 (0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1) 出现了 156 次
元组 (0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1) 出现了 227 次
元组 (0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1) 出现了 314 次
元组 (0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1) 出现了 198 次
元组 (0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0) 出现了 203 次
元组 (1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0) 出现了 179 次
元组 (1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0) 出现了 84 次
元组 (1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0) 出现了 87 次
元组 (1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0) 出现了 1 次
元组 (0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1) 出现了 2 次
元组 (1, 0

In [10]:
for index,each in enumerate(jnp.array(items)):
    value = total_machine(abnormal_params,each)
    print(f'{index}| log(Psi)={value},对应组态={each}')


0| log(Psi)=(0.9047828382909447+0.8238769604867494j),对应组态=[0 1 1 0 0 1 0 1 1 0 0 1]
1| log(Psi)=(1.0107061190308078+0.6562025777979442j),对应组态=[0 1 1 0 0 1 0 1 1 0 1 0]
2| log(Psi)=(1.0462752637873527+1.1346349343593232j),对应组态=[1 0 1 0 0 1 0 1 1 0 0 1]
3| log(Psi)=(0.9047828382909447+0.8238769604867494j),对应组态=[1 0 0 1 0 1 1 0 0 1 0 1]
4| log(Psi)=(1.0462752637873527+1.1346349343593232j),对应组态=[1 0 0 1 1 0 1 0 0 1 0 1]
5| log(Psi)=(1.0107061190308078+0.6562025777979442j),对应组态=[1 0 1 0 0 1 1 0 0 1 0 1]
6| log(Psi)=(1.0107061190308078+0.6562025777979442j),对应组态=[0 1 0 1 1 0 1 0 0 1 1 0]
7| log(Psi)=(0.9047828382909447+0.8238769604867494j),对应组态=[0 1 0 1 1 0 0 1 0 1 1 0]
8| log(Psi)=(1.0462752637873527+1.1346349343593232j),对应组态=[0 1 0 1 1 0 0 1 1 0 1 0]
9| log(Psi)=(1.0462752637873527-2.00695771923047j),对应组态=[1 0 1 0 1 0 0 1 0 1 0 1]
10| log(Psi)=(1.0107061190308078-2.485390075791849j),对应组态=[0 1 1 0 1 0 1 0 0 1 0 1]
11| log(Psi)=(0.9047828382909447-2.3177156931030436j),对应组态=[0 1 1 0 1 0 0 1 0 

In [14]:
jnp.set_printoptions(
    linewidth=9999,   # 单行宽度拉满，绝不自动换行
    threshold=jnp.inf, # 全部打印，不省略
    precision=8,      # 小数位数按需调整
    suppress=False
)

In [15]:
x1 = hi.all_states()[0]
x2 = hi.all_states()[1]
x3 = hi.all_states()[2]
x4 = hi.all_states()[3]

total_matrix_machine(abnormal_params,jnp.array([x2,x1,x4]))


Array([[ 0.74706912-0.31409474j,  0.21041874-0.45642138j, -0.08814174+0.08362629j],
       [ 0.62144677+0.16357725j,  0.45697319+0.15953518j, -0.33406952-0.37068233j],
       [ 0.60662779+1.13309419j,  0.82964123+0.19786961j, -0.27874025+0.4856571j ]], dtype=complex128)

In [16]:
total_matrix_machine(abnormal_params,jnp.array([x3,x2,x1]))

Array([[ 0.60001211+1.92829853j,  0.65105917+0.61600483j, -1.43412407-0.66089986j],
       [ 0.74706912-0.31409474j,  0.21041874-0.45642138j, -0.08814174+0.08362629j],
       [ 0.62144677+0.16357725j,  0.45697319+0.15953518j, -0.33406952-0.37068233j]], dtype=complex128)